# Tensors, shapes and linear maps

**Core time:** 90–120 minutes, plus exercises. **Prerequisites:** basic Python, sums and multiplication. **Outcomes:** translate a model expression into array shapes; compute dot products and norms; distinguish matrix multiplication from elementwise operations; diagnose broadcasting errors.

## 1. Begin with the question an array answers

Suppose a laboratory records three measurements for each of four devices. An array `X` with shape `(4, 3)` is useful only after we say what the axes mean: row = device, column = measurement. A transposed array has the same twelve values but a different interpretation. In this course, a tabular batch normally has shape `(n_samples, n_features)`. An image batch in PyTorch normally uses `(batch, channels, height, width)`. Write these conventions beside a calculation before writing code.

A scalar has no indexing axes, a vector has one, a matrix has two, and a higher-order array has more. The word tensor here means such a numerical array; it does not automatically mean that coordinate-transformation rules from differential geometry are being used. Array order is also different from matrix rank: a matrix is a two-axis object even when all its columns are multiples of one another and its rank is one.

The dimensions tell you what combinations are possible. If each observation has three features and we want two scores, choose a weight matrix with shape `(3, 2)` and a bias with shape `(2,)`. Multiplication sums over the shared feature dimension, producing `(4, 2)`. The same weights are applied to every observation; the weights do not depend on batch size.

```math
X\in\mathbb{R}^{n\times d},\quad W\in\mathbb{R}^{d\times k},\quad b\in\mathbb{R}^{k},\qquad Z=XW+\mathbf{1}b^\top.
```

Each output `Z[i,j]` is the dot product of input row `i` and weight column `j`, followed by a bias. This is a linear map plus a translation, usually called an affine layer. A strictly linear map sends zero to zero. The bias makes the layer affine unless it is zero. Later, nonlinear activation functions allow stacked layers to express more than one affine transformation.

## 2. Geometry supplies checks that shapes cannot

For vectors `u` and `v`, the dot product adds products of matching coordinates. The Euclidean norm is the square root of the dot product of a vector with itself. A vector can have a small norm yet a large effect on a sensitive model; a norm describes a perturbation size, not its consequence. Other useful norms are the sum of absolute values (L1) and largest absolute coordinate (L-infinity). The allowed perturbation set in a robustness experiment depends on which norm is used.

If both vectors are nonzero, cosine similarity divides their dot product by the product of their Euclidean norms. It measures direction agreement and ignores positive rescaling. It is undefined for a zero vector. Code must decide explicitly how to handle that case rather than quietly divide by a tiny constant and describe the result as a true cosine.

For `u=[3,4]` and `v=[4,-3]`, the dot product is `12-12=0`, both norms are 5, and the vectors are orthogonal. For `2*u`, the norm doubles but the cosine with `u` remains one. These simple invariants are useful tests for an embedding or normalization implementation.

## 3. Worked example: two devices, two outputs

Take `X=[[1,2],[3,4]]`, `W=[[2,-1],[0,3]]` and `b=[1,-2]`. For the first observation, output 1 is `1*2 + 2*0 + 1 = 3`; output 2 is `1*(-1) + 2*3 - 2 = 3`. For the second observation the outputs are 7 and 7. Therefore `Z=[[3,3],[7,7]]`.

The elementwise operation `X*W` also happens to be legal because both shapes are `(2,2)`, but it produces `[[2,-2],[0,12]]`. This is a particularly dangerous bug: there is no shape exception. Always test a tiny case whose answer you know, then test a non-square case so accidental shape coincidences cannot rescue a wrong implementation.



In [ ]:
import numpy as np
X = np.array([[1., 2.], [3., 4.]])
W = np.array([[2., -1.], [0., 3.]])
b = np.array([1., -2.])
Z = X @ W + b
assert Z.shape == (2, 2)
np.testing.assert_allclose(Z, [[3., 3.], [7., 7.]])
u = np.array([3., 4.])
v = np.array([4., -3.])
assert u @ v == 0
assert np.linalg.norm(u) == 5
print(Z)



The `@` operator expresses a matrix product. Floating-point assertions should generally use a tolerance; exact equality is appropriate here only for the deliberately small integer-representable calculations. Later gradient calculations accumulate rounding error.

## 4. Broadcasting is useful and capable of lying to you

NumPy compares dimensions from the right: dimensions are compatible when they are equal or one is 1; missing leading dimensions behave as 1. Thus a bias of shape `(k,)` can be added to an `(n,k)` batch. But subtracting targets with shape `(n,1)` from predictions with shape `(n,)` produces an `(n,n)` matrix, not a length-n residual. A loss averaged over this matrix is a number and can even decrease during training, while measuring the wrong thing.

Inspect `prediction.shape == target.shape` before a regression loss. Use `keepdims=True` when a reduction must preserve an axis. `X.mean(axis=0)` averages observations for each feature; `X.mean(axis=1)` averages features within each observation. These are different statistical questions. Avoid using `reshape` as a cure for a semantic mismatch: it changes interpretation without checking what the axes mean.

## 5. Guided laboratory

Run `python lab01_math.py --output outputs/math`. Find the shape and gradient checks in its result file. Then create three observations with four features and a `(4,2)` weight matrix. Compute one output by hand and compare it to the program. Add a test that intentionally supplies a `(3,1)` target to a `(3,)` prediction and rejects it before subtraction. Finally, normalize two nonzero vectors and verify their lengths are one. Keep the original vectors so that normalization does not silently modify data used in a later calculation.

## Exercises

1. **Shapes:** A batch contains 16 observations with 7 features; the layer has 5 outputs. State the shapes of weights, bias, output and output-gradient. How many trainable scalar parameters are present?
2. **Reasoning:** Show algebraically why two affine layers with no intervening nonlinearity are equivalent to a single affine layer. Does adding depth alone increase their function class?
3. **Debugging:** Explain why `(5,) - (5,1)` gives `(5,5)`. Construct a case where the incorrect loss is zero and discuss why one passing example is weak evidence.
4. **Extension:** Compare L1, L2 and L-infinity norms of `[1,-2,2]`. Which perturbations can share an L-infinity budget while having different L2 sizes?

## Self-check

1. Weights `(7,5)`, bias `(5,)`, output and output-gradient `(16,5)`; parameters `7*5+5=40`. 2. `(XW1+b1)W2+b2 = X(W1W2)+(b1W2+b2)`, so the composition is affine. 3. The prediction expands across one axis and target across the other; all predictions and targets equal to the same constant hide the bug. Test unequal values as well. 4. The norms are 5, 3 and 2. A one-coordinate change of size 2 and a three-coordinate change of size 2 each both satisfy the same L-infinity budget but have different Euclidean lengths.

## Primary reading

- [NumPy: broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html): check the right-to-left compatibility rule against your residual example.
- [Goodfellow, Bengio and Courville: Linear Algebra, Chapter 2](https://www.deeplearningbook.org/contents/linear_algebra.html): use the notation and norm sections as a reference, then restate each in your own words.


# Least squares, rank, SVD and principal components

**Core time:** 120 minutes. **Prerequisites:** tensor shapes and dot products. **Outcomes:** interpret least squares as projection; recognize rank deficiency; use a numerical solver instead of an inverse; connect singular values to compression and PCA.

## 1. An imperfect equation can still have a best approximation

We would like to fit a line to three observations: at `x=0,1,2`, the measured responses are `1,2,2`. A line has only two parameters, intercept and slope, so it generally cannot pass through every noisy observation. Put a column of ones and the x-values into a design matrix `A`; let `theta=[intercept,slope]`. The vector `A@theta` lies in the space spanned by the columns of `A`. Least squares chooses the point in that space closest to the observed response vector.

```math
\hat\theta=\arg\min_\theta\frac12\|A\theta-y\|_2^2,\qquad
A^\top(A\hat\theta-y)=0.
```

The second expression says the residual is perpendicular to every column of the design matrix. It follows by differentiating the squared error, but also has a geometric explanation: if a residual component pointed along a column direction, moving slightly in that direction could reduce the distance. This condition is an excellent implementation check. It is not the same as requiring every residual to be zero.

## 2. Work the example all the way through

For the three data points, `A.T@A=[[3,3],[3,5]]` and `A.T@y=[5,6]`. The two normal equations are `3a+3b=5` and `3a+5b=6`. Subtracting them gives `2b=1`, so the slope is 0.5 and intercept is 7/6. Predictions are `[7/6,10/6,13/6]`; residuals `prediction-y` are `[1/6,-1/3,1/6]`. They sum to zero and their dot product with `[0,1,2]` is zero. The sum of squared residuals is 1/6.

These normal equations are valuable for a hand derivation. They are not an instruction to implement `inv(A.T@A)@A.T@y`. Forming the normal matrix squares the 2-norm condition number for a full-column-rank matrix, worsening numerical sensitivity. A least-squares routine using appropriate matrix factorization is usually preferable.



In [ ]:
import numpy as np
A = np.column_stack([np.ones(3), np.arange(3.)])
y = np.array([1., 2., 2.])
theta, residuals, rank, singular_values = np.linalg.lstsq(A, y, rcond=None)
r = A @ theta - y
np.testing.assert_allclose(theta, [7/6, 1/2])
np.testing.assert_allclose(A.T @ r, 0, atol=1e-12)
np.testing.assert_allclose(r @ r, 1/6)
print(theta, rank, singular_values)



The solver also reports rank and singular values. Do not interpret an empty `residuals` array from this API as proof of zero error; compute `A@theta-y` directly when you need residuals, because the returned summary depends on dimensions and rank.

## 3. Rank asks how many independent directions exist

If one feature is exactly twice another, its column adds no independent direction. Many parameter vectors can then produce the same fitted values. A solver can return a minimum-norm solution, but this does not make each coefficient independently identifiable. If columns are almost dependent, small changes in data can produce large coefficient changes even when predictions barely change.

Scale also matters. A feature measured in millions and another measured in tiny fractions create very different numerical scales. Feature scaling should use training data only. Ridge regularization adds a penalty on coefficient magnitude, stabilizing some ill-conditioned problems while introducing bias. If the intercept is not intended to be penalized, represent that choice explicitly; blindly adding an identity matrix penalizes every coordinate.

```math
\hat\theta_\lambda=\arg\min_\theta\frac12\|A\theta-y\|_2^2+\frac{\lambda}{2}\|\theta\|_2^2.
```

For strictly positive lambda this penalized objective has a unique solution when all coordinates are penalized. That numerical property is separate from whether the modeling assumptions are appropriate. Select regularization using validation data, not by making the test coefficient story attractive.

## 4. SVD reveals the directions a matrix preserves

The singular value decomposition factors a matrix into orthogonal directions and nonnegative scaling values. In a reduced decomposition, a matrix with shape `(m,n)` has `U` of shape `(m,r)`, a length-r vector of singular values, and `Vt` of shape `(r,n)`, where `r=min(m,n)` before discarding zero directions. Some libraries' “full” decomposition returns larger orthogonal factors, so state which form you use.

```math
A=U\Sigma V^\top,\qquad A_k=U_{:,1:k}\Sigma_{1:k,1:k}V_{:,1:k}^\top.
```

Retaining the largest k singular values gives a best rank-k approximation in Frobenius norm. The squared reconstruction error is the sum of squares of discarded singular values. For `diag(5,2,1)`, a rank-one approximation retains 5 and has squared error `2^2+1^2=5`. Rank two has squared error 1. This does not guarantee a classifier retains accuracy after its weight matrix is compressed: the matrix error and prediction error are different objectives.

## 5. PCA is a data transformation with a fitting step

Center training features by subtracting their training means, then take the right singular vectors of the centered matrix. Project observations onto the leading vectors to obtain principal components. With the usual sample-covariance convention and `n>1`, component variances equal squared singular values divided by `n-1`. If features use incompatible units, decide whether training-only standardization is appropriate before PCA.

PCA maximizes retained variance under its orthogonality constraints; it does not maximize class separation. A low-variance direction can carry important predictive information. Fitting PCA on the complete dataset before splitting leaks test-distribution information into preprocessing. Fit the mean and components only on training observations, and apply the same transformation to validation and test sets.

## Guided laboratory and exercises

Run the SVD section of `lab01_math.py` and inspect its reconstruction-error checks. Then complete these tasks:

1. Duplicate the slope column in the worked regression. Report the rank and explain why coefficients may change while fitted values do not.
2. For `diag(8,3,1,0)`, calculate the rank, rank-two squared error and fraction of squared singular-value energy retained at rank two.
3. Create a two-feature classification problem where the informative feature has very small variance and the irrelevant feature has large variance. Explain why raw PCA may discard the informative feature.
4. Compare solutions for a nearly dependent design under tiny response perturbations. Report both coefficient change and prediction change; do not infer instability solely from one coefficient.

## Self-check

1. Rank remains two; the duplicated features share an effect that is not uniquely attributable. 2. Rank three, squared error one, retained energy `73/74`. 3. PCA follows variance rather than labels; scaling changes the geometry but does not make PCA supervised. 4. Ill conditioning can amplify coefficient variation, while similar fitted predictions remain possible. The report should distinguish those quantities and state the numerical tolerance used to estimate rank.

## Primary reading

- [MIT 18.06SC: Singular Value Decomposition](https://ocw.mit.edu/courses/18-06sc-linear-algebra-fall-2011/pages/positive-definite-matrices-and-applications/singular-value-decomposition/): study the geometry of the factorization rather than memorizing a decomposition recipe.
- [NumPy: `linalg.lstsq`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html): inspect the return-value and rank-cutoff definitions before interpreting solver output.


# Gradients, matrix calculus and the chain rule

**Core time:** 120 minutes. **Prerequisites:** affine layers, squared error and elementary derivatives. **Outcomes:** derive a batch gradient with correct dimensions; trace a computation graph; verify an analytic gradient numerically; explain what automatic differentiation does and does not check.

## 1. A derivative is a local prediction of change

For a scalar function, the derivative predicts the change in output caused by a sufficiently small change in input. For a scalar loss depending on a parameter vector, the gradient collects the partial derivatives in the same shape as the parameters. Its dot product with a small displacement approximates the loss change. Moving in the negative-gradient direction is locally downhill when the gradient is nonzero and the step is sufficiently small. A huge step can increase the loss despite having the correct direction.

For a vector-valued function, the Jacobian organizes how each output changes with each input. Writing the full Jacobian is often unnecessary: backpropagation repeatedly applies vector–Jacobian products to carry the derivative of one scalar loss back through a graph. This is why a training framework can differentiate a model with millions of parameters without constructing every possible derivative matrix.

## 2. Derive a loss before coding it

Let `X` have shape `(n,d)`, `w` shape `(d,)`, and `y` shape `(n,)`. Define residual `r=Xw-y` and the half mean-squared loss below. The factor one-half cancels the derivative's factor two; the factor `1/n` controls the scale relative to batch size.

```math
L(w)=\frac{1}{2n}\sum_{i=1}^n(x_i^\top w-y_i)^2,
\qquad \nabla_w L=\frac{1}{n}X^\top(Xw-y),
\qquad \nabla_w^2 L=\frac{1}{n}X^\top X.
```

To derive one coordinate, differentiate each squared residual with respect to `w_j`: it contributes `r_i*x_ij/n`. Summing over observations gives component j of `X.T@r/n`. Check dimensions: `(d,n)@(n,)` gives `(d,)`, exactly the parameter shape. The Hessian is positive semidefinite because for any vector v, `v.T@X.T@X@v/n = ||Xv||²/n >= 0`. It is positive definite when the design has full column rank. This explains convexity and when the unregularized optimum is unique.

For a multi-output affine layer `Z=XW+b`, suppose `G` is the derivative of the scalar loss with respect to `Z`, shape `(n,k)`. Then the weight gradient is `X.T@G`, the bias gradient sums G over the batch, and the input gradient is `G@W.T`. If the loss already averages observations, do not divide by n again in the layer gradient.

## 3. Worked example: a complete backward pass

Use `X=[[1,2],[3,4]]`, `y=[1,2]` and `w=[0.2,-0.1]`. Predictions are `[0,0.2]`, residuals are `[-1,-1.8]`, and the half mean-squared loss is `(1+3.24)/4=1.06`. The gradient is `[(-1-5.4)/2,(-2-7.2)/2]=[-3.2,-4.6]`. A step of size 0.01 produces `w=[0.232,-0.054]`. Notice that both weights increase because both gradient coordinates are negative.



In [ ]:
import numpy as np
X = np.array([[1., 2.], [3., 4.]])
y = np.array([1., 2.])
w = np.array([0.2, -0.1])
def loss(v):
    r = X @ v - y
    return float(r @ r / (2 * len(y)))
analytic = X.T @ (X @ w - y) / len(y)
h = 1e-5
numeric = np.zeros_like(w)
for j in range(len(w)):
    direction = np.zeros_like(w)
    direction[j] = h
    numeric[j] = (loss(w + direction) - loss(w - direction)) / (2*h)
np.testing.assert_allclose(analytic, [-3.2, -4.6])
np.testing.assert_allclose(analytic, numeric, rtol=1e-6, atol=1e-8)
assert loss(w - 0.01*analytic) < loss(w)
print(loss(w), analytic, numeric)



Central differences use two loss evaluations per coordinate. They are useful for small checks, not an efficient replacement for reverse-mode differentiation of a large network. Make the check deterministic and use float64. Dropout, random augmentation and changing minibatches otherwise cause changes unrelated to the derivative you are checking.

## 4. Chain rule through a nonlinearity

Consider `z=w*x+b`, `a=max(0,z)`, and `L=(a-y)^2/2`. If `z>0`, the derivative through ReLU is one; if `z<0`, it is zero. At zero the classical derivative is undefined and implementations choose a convention. For `x=2,w=0.5,b=0,y=3`, z and a are one, the derivative of loss with respect to a is -2, the weight derivative is -4 and bias derivative -2. Change the bias to -2 and the preactivation becomes -1: this example's ReLU derivative is zero even though the prediction is poor.

Automatic differentiation follows the operations actually executed. If a programmer detaches a tensor, converts it to a plain Python number or accidentally evaluates the wrong residual, autograd cannot infer the intended model. A correct derivative of an incorrect computation is still an incorrect experiment. Inspect tensor shapes, gradient presence and the objective definition separately.

## 5. Gradient-check failure modes

Very large finite-difference steps introduce approximation error; extremely small steps lose information through subtraction of nearly equal numbers. Try a range such as `1e-3,1e-4,1e-5,1e-6` in float64. Avoid points at a nonsmooth activation boundary. Compare absolute error near a zero gradient and relative error for larger gradients; a relative error with a nearly zero denominator can be misleading. A scalar loss may be averaged over examples, outputs or both; record which convention was differentiated.

## Exercises and self-check

1. Add `lambda*||w||²/2` to the worked loss. Derive the additional gradient and Hessian. **Check:** add `lambda*w` and `lambda*I` respectively.
2. For `X` of shape `(8,3)`, `W` `(3,4)` and G `(8,4)`, state the shapes of all three affine-layer gradients. **Check:** `(3,4)`, `(4,)` and `(8,3)`.
3. A finite-difference check fails only near ReLU zero. Design a diagnostic before concluding autograd is broken. **Check:** move away from the kink, test a smooth function, vary step size and verify deterministic inputs.
4. Implement a two-layer network with tanh, derive its gradients and compare with both numerical differences and PyTorch autograd. Submit the maximum coordinate error and the tested input/parameter shapes, not only “passed.”

Run `lab01_math.py` for the executable linear-gradient check; the PyTorch module later extends the same reasoning to an MLP.

## Primary reading

- [PyTorch: automatic differentiation](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html): connect graph construction and accumulated gradients to the derivation above.
- [Goodfellow, Bengio and Courville: Numerical Computation](https://www.deeplearningbook.org/contents/numerical.html): read the gradient-based optimization discussion and identify its smoothness assumptions.


# Probability, Bayes' rule and likelihood

**Core time:** 100–120 minutes. **Prerequisites:** fractions, sums and logarithms. **Outcomes:** distinguish a random variable from an observed value; compute conditional probabilities; use Bayes' rule without ignoring base rates; connect maximum likelihood to classification losses.

## 1. Specify what is random

A random variable maps possible outcomes to values. A dataset records realized values; a probability model represents uncertainty about outcomes under stated assumptions. The same symbol is often used for both in papers, so write down whether `Y` means a random label or `y_i` means an observed label. A probability mass function assigns probabilities to discrete values. A density for a continuous variable integrates to probability over an interval; the density itself can exceed one and the probability of an exact real-valued point is ordinarily zero under a continuous distribution.

For a discrete variable, expectation is a probability-weighted sum. Variance is expected squared deviation from the mean. The expectation of a sum equals the sum of expectations even without independence. Variances add only after accounting for covariance. For two variables, the variance of their sum includes twice their covariance. Correlation measures normalized linear association; zero correlation alone does not imply independence.

## 2. Conditional probability changes the reference population

The quantity `P(A|B)` asks about A among outcomes satisfying B. It is defined by `P(A and B)/P(B)` when the denominator is positive. Multiplying this identity in two orders gives Bayes' rule. The denominator is not a decoration: it ensures that the posterior probabilities sum to one and accounts for every way the observation could occur.

```math
P(H\mid E)=\frac{P(E\mid H)P(H)}{P(E)},\qquad
P(E)=\sum_h P(E\mid H=h)P(H=h).
```

Independence means knowing one event does not change the probability of another. Conditional independence can hold given a third variable even when marginal independence does not. Many useful models deliberately assume conditional independence to simplify estimation; this is a modeling assumption, not something guaranteed by data being in different columns.

## 3. Worked example: a rare-event detector

Imagine a synthetic equipment-fault population in which 1% of devices are faulty. A detector flags 90% of faults and falsely flags 5% of healthy devices. Among 10,000 devices, expect 100 faults and 9,900 healthy devices. The detector flags about 90 faulty and 495 healthy devices. Thus the fraction of flagged devices that are faulty is `90/(90+495)`, approximately 15.38%.

This is not the same as the 90% detection rate. The latter conditions on actual faults; the former conditions on flags. They answer different questions. If deployment prevalence changes while those class-conditional detection rates remain fixed, the precision changes. A reported accuracy or recall alone cannot tell an operator how many flags to expect or how useful they will be.



In [ ]:
prevalence = 0.01
tpr = 0.90
fpr = 0.05
flag_probability = tpr*prevalence + fpr*(1-prevalence)
posterior = tpr*prevalence / flag_probability
assert abs(posterior - 2/13) < 1e-12
print(f"P(fault | flag) = {posterior:.4f}")



The numerical population is an original teaching example, not an estimate of any real device or medical system. When presenting a probability calculation, disclose where every input rate came from and which rates you assumed remain stable.

## 4. Likelihood holds the observations fixed

A probability model gives `p(data|theta)`. Once data are observed, the same expression viewed as a function of theta is the likelihood. It is not a probability distribution over theta unless a prior and a normalizing calculation turn it into a posterior. Maximum likelihood selects the parameter value that assigns the largest model probability to the observations.

For independent Bernoulli observations with success probability p, the likelihood is a product of p for successes and `1-p` for failures. Its negative log is a sum. With three successes in four trials, differentiating the log likelihood yields a maximum at p=3/4. Independence and a shared p were assumptions; repeated measurements of one changing system may violate them.

```math
-\log L(p)=-\sum_i\left[y_i\log p+(1-y_i)\log(1-p)\right].
```

When p is generated by a logistic model for each input, the average of these terms becomes binary cross-entropy. For multiple mutually exclusive classes, a categorical model gives the negative log probability of the observed class. The logarithm makes products numerically manageable and turns independent contributions into sums. It does not make an incorrectly specified model true.

## 5. From logits to stable probabilities

Softmax transforms real-valued logits into positive values summing to one. Adding the same constant to all logits leaves the probabilities unchanged. Subtracting the maximum before exponentiating uses that invariance to avoid overflow. A library's cross-entropy function often accepts raw logits and performs a stable log-softmax internally; applying softmax first can be both unnecessary and numerically harmful.



In [ ]:
import numpy as np
logits = np.array([1000., 1001., 999.])
shifted = logits - logits.max()
p = np.exp(shifted) / np.exp(shifted).sum()
assert np.isfinite(p).all()
np.testing.assert_allclose(p.sum(), 1.)
loss = -shifted[1] + np.log(np.exp(shifted).sum())
print(p, loss)



## Exercises

1. Recalculate the detector's precision at prevalence 10%, keeping TPR and FPR fixed. Explain which assumption may fail in real deployment.
2. For a Bernoulli variable with success probability p, derive its mean and variance from `Y²=Y`.
3. A pair of measurements has variance 4 each and covariance 3. What is the variance of their average? Compare with the false assumption of independence.
4. Explain why a confident probability is not necessarily calibrated. Design an empirical check using repeated predictions and observed outcomes.

## Self-check

1. `0.09/(0.09+0.045)=2/3`; conditional detector performance can change across populations. 2. Mean p, variance `p-p²`. 3. `(4+4+2*3)/4=3.5`; independence would give 2. 4. Confidence is a model output; calibration concerns agreement with observed frequencies on a specified population. A reliability plot needs adequate sample counts and should be assessed on data not used to fit the calibrator.

## Primary reading

- [Goodfellow, Bengio and Courville: Probability and Information Theory](https://www.deeplearningbook.org/contents/prob.html): use the conditional-probability, expectation and information-measure sections as definitions, then derive the examples independently.
- [PyTorch: CrossEntropyLoss](https://docs.pytorch.org/docs/2.9/generated/torch.nn.CrossEntropyLoss.html): inspect the expected logits/target shapes and reduction convention.


# Estimation, uncertainty and honest comparisons

**Core time:** 120 minutes. **Prerequisites:** expectation, variance and conditional probability. **Outcomes:** identify an estimand and unit of analysis; separate sampling variability from training variability; compute a paired comparison; interpret an interval without turning it into a probability claim about a fixed parameter.

## 1. State what you want to estimate

“Model quality” is too vague to estimate. A more precise estimand is the expected accuracy of a fixed trained model on future independent observations from a specified population. Another is the expected performance of an entire training procedure, averaged over new training samples and random initialization. A third is the average per-client performance in a federation. These are different quantities, even when each is reported as a percentage.

The observed average is an estimator, and the particular value computed from one dataset is an estimate. Bias concerns the estimator's expectation across repeated samples; variance concerns its spread. A smaller standard error does not repair selection bias, measurement error or a mismatch between the evaluation population and deployment. More duplicated observations do not supply the same information as more independent subjects.

## 2. The unit of independence determines the uncertainty calculation

Suppose ten measurements are collected from each of twenty devices. There are 200 rows but only twenty independently sampled devices. Resampling rows as if all 200 were independent can produce overly narrow intervals. If the target is performance on new devices, keep device groups intact during splitting and resample devices for uncertainty analysis. In time series, neighboring observations may be dependent; an appropriate block or time-aware method requires assumptions about the process.

For independent Bernoulli correctness indicators with a fixed classifier, an approximate standard error for accuracy is `sqrt(p_hat*(1-p_hat)/n)`. At 90 correct out of 100, that is 0.03. The familiar normal approximation gives roughly 0.90 ± 0.0588. This approximation can be poor for small n or extreme proportions; use a justified binomial interval such as Wilson's when appropriate. Do not silently reuse this formula for correlated client records or for a model chosen by repeatedly inspecting the same test set.

## 3. Compare models on paired observations

When two fixed classifiers predict the same examples, their errors are related. Form a difference in correctness for each example: +1 if only B is correct, -1 if only A is correct, and 0 if they agree in correctness. The mean difference is B's accuracy minus A's. Resample paired indices so that each example's two predictions stay together. Separately resampling the models destroys pairing and answers a different question.

Imagine 100 examples: both models are correct on 80, only A on 5, only B on 10, and both wrong on 5. A has accuracy 0.85 and B 0.90. Only 15 examples distinguish them; the observed five-point improvement should not be interpreted as certain superiority on all future data. An interval around the paired difference communicates finite-sample uncertainty under the resampling assumptions.



In [ ]:
import numpy as np
a = np.array([1]*80 + [1]*5 + [0]*10 + [0]*5)
b = np.array([1]*80 + [0]*5 + [1]*10 + [0]*5)
d = b - a
rng = np.random.default_rng(17)
indices = rng.integers(0, len(d), size=(5000, len(d)))
bootstrap_differences = d[indices].mean(axis=1)
lo, hi = np.quantile(bootstrap_differences, [0.025, 0.975])
np.testing.assert_allclose(d.mean(), 0.05)
print({"difference": d.mean(), "percentile_interval": [lo, hi]})



This percentile bootstrap is an accessible teaching method, not a universal optimal interval. State the number of resamples, random seed, resampling unit and interval construction. Do not treat a rounded boundary near zero as a verdict that the models are identical or different in every practical sense.

## 4. Training seeds answer another question

Training the same architecture with seeds 1, 2 and 3 probes variation from the stochastic training process conditional on the chosen data split. It does not create three independent test datasets. Report per-seed results as well as mean and standard deviation. Three seeds can reveal gross instability, but do not support a precise estimate of the full performance distribution. If algorithms share split and seed schedules, analyze paired per-run differences while explaining what aspects are genuinely matched.

Repeatedly trying models until one exceeds a target on the test set turns that set into development data. Even if every individual confidence interval is calculated correctly, selection can invalidate a naive interpretation of the winning result. Use a validation set for exploration, freeze the decision rule, and evaluate the reserved test set once for the final report. If you must revisit it, disclose the reuse and seek a new independent holdout rather than pretending it remained untouched.

## 5. Interpretation belongs beside the interval

A frequentist 95% confidence procedure has a coverage property across repeated samples under its assumptions. For a realized interval and a fixed parameter, “there is a 95% probability that the parameter is in this interval” is not its standard interpretation. A Bayesian credible interval is a different object based on an explicit prior and likelihood. Neither interpretation licenses ignoring distribution shift.

Separate statistical detectability from practical importance. A tiny improvement measured very precisely may not justify twice the inference latency. A practically large change may remain uncertain with a small sample. State a meaningful effect size or decision threshold before seeing results, then discuss both estimated benefit and uncertainty.

## Exercises and self-check

1. Explain why bootstrapping individual images is inappropriate when five images come from each independently sampled patient and deployment concerns new patients. **Check:** resample patients/groups and keep related images together; the medical context is only an independence example, not a deployment recommendation.
2. A report gives “0.92 ± 0.01.” List missing information. **Check:** metric, denominator, interval versus standard deviation, confidence level, number of runs/samples, resampling unit, split and selection procedure.
3. Run the paired example with 500, 5,000 and 20,000 bootstrap resamples. What uncertainty decreases as this number grows? **Check:** Monte Carlo error of the interval calculation, not uncertainty caused by having only 100 original observations.
4. Design a table for three seeds and two models that preserves pairing. Include an effect-size column and a statement of the experiment's limited scope.

## Primary reading

- [SciPy: `stats.bootstrap`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html): inspect paired resampling, interval methods and degeneracy warnings.
- [scikit-learn: cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html): choose a splitter according to groups, time and the intended generalization question.


# Optimization: curvature, stochastic gradients and adaptive steps

**Core time:** 120 minutes. **Prerequisites:** gradients, expectations and matrix rank. **Outcomes:** analyze gradient descent on a quadratic; distinguish an objective from an optimizer; explain minibatch noise; compare momentum and Adam without claiming universal superiority.

## 1. Separate the destination from the route

An objective specifies what is minimized. An optimizer specifies how parameters are updated. Regularization changes the objective or update rule; learning-rate scheduling changes the route taken. A validation metric may differ from the training objective. A decrease in training cross-entropy does not guarantee an improvement in every fairness metric, every class recall or deployment latency.

For differentiable `f(w)`, gradient descent updates `w` by subtracting a positive learning rate times the gradient. A first-order approximation predicts a decrease proportional to the squared gradient norm, but it ignores curvature. If curvature is large, the approximation is valid only over a smaller step. This is why “the gradient points downhill” does not justify an arbitrary learning rate.

```math
w_{t+1}=w_t-\eta\nabla f(w_t).
```

Convexity means every line segment between two domain points lies above the function graph in the appropriate inequality. A differentiable convex function has no strictly worse local minimum than its global minima. Strict or strong convexity adds stronger uniqueness or curvature properties. A neural network's parameter objective is generally nonconvex; successful optimization does not establish global optimality.

## 2. Worked example: see instability before running a network

Take `f(w)=2w²`, gradient `4w`, and start at w=1. The update is `w_next=(1-4*eta)*w`. Convergence to zero requires the magnitude of this multiplier to be below one, giving `0<eta<0.5`. With eta=0.1, successive values are 1, 0.6, 0.36 and 0.216. With eta=0.6, they are 1, -1.4, 1.96 and -2.744; alternating signs do not prevent divergence.

For a positive-definite quadratic with Hessian H, each eigen-direction behaves similarly. A sufficient fixed-step range is `0<eta<2/lambda_max(H)`. A large ratio between largest and smallest eigenvalues produces narrow directions that restrict the step while broad directions converge slowly. Feature scaling can improve this conditioning. It does not remove all challenges of a nonconvex network.



In [ ]:
import numpy as np
def trajectory(eta, steps=8):
    w = 1.0
    values = [w]
    for _ in range(steps):
        w -= eta * 4*w
        values.append(w)
    return np.array(values)
assert abs(trajectory(0.1)[-1]) < 0.02
assert abs(trajectory(0.6)[-1]) > 1
print(trajectory(0.1))
print(trajectory(0.6))



## 3. A minibatch gradient estimates a full-data gradient

If examples are sampled uniformly and the per-example gradients are averaged, the minibatch gradient is an unbiased estimator of the empirical full-data gradient under the sampling assumptions. This is a statement about the empirical objective, not a guarantee about the population objective. Biased sampling, incorrectly weighted losses or stateful transformations can change the relationship.

Larger batches often reduce gradient noise but require more memory and change the number of updates per epoch. Comparing two batch sizes at equal epochs does not equalize update count. Comparing at equal steps does not equalize examples processed. State the budget: examples, optimizer updates, wall-clock time or energy. An experiment cannot claim these are all held fixed unless they actually are.

Shuffling without replacement is common but is not identical to drawing independent samples with replacement. For a short teaching run this distinction may have modest practical consequences; in a theoretical claim, the sampling convention is part of the assumptions. Never turn a convenient approximation into an unstated theorem.

## 4. Momentum and Adam change how history is used

One momentum convention forms a velocity `v=beta*v+g` and updates `w-=eta*v`. Another uses `(1-beta)*g`, changing the scaling. Check the exact library convention before matching a paper's hyperparameters. Momentum can smooth oscillations but can also overshoot after an abrupt change.

Adam tracks exponential averages of gradients and squared gradients, corrects their initial bias, and divides the first-moment estimate by a square root of the second-moment estimate plus epsilon. The denominator is coordinatewise; it is not the full inverse Hessian. Epsilon prevents problematic division and affects behavior near very small values. Adam is not a proof of convergence for any arbitrary loss or learning rate.

```math
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,\quad
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\qquad
w_{t+1}=w_t-\eta\frac{m_t/(1-\beta_1^t)}{\sqrt{v_t/(1-\beta_2^t)}+\epsilon}.
```

L2 regularization and decoupled weight decay are not generally equivalent under adaptive optimizers. Record whether you used Adam, AdamW or another implementation, and which parameters receive decay. Biases and normalization parameters are often treated differently; if so, describe it rather than relying on an unstated convention.

## 5. Practical debugging sequence

First verify finite inputs, labels, loss and gradients. Next overfit a tiny batch using a simple model. Inspect whether gradients are absent, identically zero or extremely large. Check that gradients are cleared at the intended time, that the optimizer updates the model's actual parameters and that evaluation runs in evaluation mode. Only then compare optimizers or elaborate schedules. Gradient clipping limits a chosen norm; it does not repair a sign error, target leakage or a detached computation graph.

## Exercises and self-check

1. Analyze `f(w)=5w²/2`. What learning-rate interval converges? **Check:** gradient 5w; `0<eta<0.4`.
2. Two runs use batch sizes 32 and 128 for ten epochs on 1,024 examples. Count updates assuming no remainder. **Check:** 320 versus 80; this is not an equal-update comparison.
3. Why can a noisy training curve still reflect useful learning? **Check:** minibatch losses vary with sampled examples; inspect a consistently evaluated full validation metric and smoothed training diagnostics without hiding raw data.
4. Implement gradient descent for a diagonal quadratic with eigenvalues 1 and 100. Compare trajectories before and after a variable rescaling. Report the learning rate and stopping rule; do not only plot the easier case.

Run `lab01_math.py` for stable and unstable quadratic trajectories. The full assignment asks you to connect these observations to a checked gradient and a declared numerical tolerance.

## Primary reading

- [Boyd and Vandenberghe: Convex Optimization](https://web.stanford.edu/~boyd/cvxbook/): consult convexity and unconstrained minimization; distinguish their assumptions from neural-network training.
- [Kingma and Ba: Adam](https://arxiv.org/abs/1412.6980): read the update definition and bias correction, then compare it with your implementation.
- [PyTorch: AdamW](https://docs.pytorch.org/docs/2.9/generated/torch.optim.AdamW.html): inspect decay semantics and defaults before configuring experiments.


# Mathematical foundations laboratory: from equations to checked evidence

**Core time:** 2–3 hours. **Prerequisites:** Math 1–6. **Outcomes:** organize a numerical experiment; choose meaningful assertions; connect numerical conditioning, gradient correctness and uncertainty; prepare the first assessed artifact.

## 1. The scientific question

The laboratory asks: “Can I explain and verify the numerical computations underlying a simple learning algorithm?” It deliberately uses small matrices and synthetic observations. Small examples make it possible to inspect every intermediate quantity, which is valuable before scaling to a neural network. Passing these examples does not prove a general library correct, but failing them reveals a concrete inconsistency to investigate.

Download the [laboratory bundle](https://www.sail-research.com/training/materials/sail-training-labs.zip), follow the setup guide, and run `python lab01_math.py --output outputs/math-baseline`. Preserve the output JSON and the terminal output. The program checks an affine map, a least-squares solution, a gradient, a truncated SVD and stable versus unstable optimization. It also records a paired-bootstrap demonstration. Read the code before changing its inputs.

## 2. Floating-point arithmetic is a model of computation

Real arithmetic on paper and finite-precision arithmetic on a machine are different. A floating-point format represents a finite set of values. Rounding error can accumulate, and subtracting nearly equal quantities can lose relative accuracy. Associativity need not hold exactly: `(a+b)+c` can differ from `a+(b+c)`. This does not mean the computer is random; it means the operation order and representation matter.



In [ ]:
import numpy as np
a = np.float64(1e16)
b = np.float64(-1e16)
c = np.float64(1.)
print((a+b)+c, a+(b+c))
small = np.array([1e-8, 2e-8], dtype=np.float64)
assert np.isfinite(small).all()



Do not use an unnecessarily large tolerance to make a failed test pass. Relate the tolerance to scale, dtype, conditioning and the operation being checked. An absolute tolerance is useful near zero; a relative tolerance scales with the expected magnitude. A result that is “close” under a loose threshold may still be scientifically unusable if the effect of interest is smaller than the numerical error.

## 3. Write tests that would catch a plausible mistake

For matrix multiplication, test a hand-computed non-square example and reject incompatible feature dimensions. For a least-squares solution, test residual orthogonality and compare against a trusted solver. For a gradient, compare analytic and central-difference values at several nontrivial parameter points. For SVD, check reconstruction and that rank-k approximation error matches the discarded singular values within tolerance. For probabilities, check finite values, nonnegativity and unit sum.

These are semantic tests: they connect code to a mathematical claim. A test that merely reproduces the same implementation formula can share its bug. For example, copying `X*r` into both the implementation and expected-value calculation does not independently verify matrix multiplication. Use a tiny manually computed answer, an invariant or an independently implemented method.

## 4. Guided investigation: conditioning and optimization

Generate a two-feature regression problem where the second feature is almost a multiple of the first. Use a fixed random generator and record its seed. Solve the original problem and a version with a tiny perturbation to y. Measure both the norm of the parameter change and the norm of the prediction change. Compare with a ridge solution over a small, declared lambda grid. Choose lambda using validation data if making a predictive claim; for this numerical demonstration you may show all values without declaring a winning predictive method.

Next scale one feature by 1,000. Compute the Hessian of the half mean-squared objective, estimate its largest eigenvalue, and choose a step smaller than `2/lambda_max`. Compare a deliberately excessive step and a safe step. Plot loss against updates and state whether feature scaling changes the underlying fitted prediction family or merely its parameterization and optimization behavior.

## 5. Guided investigation: stable log probabilities

Compare naive softmax with max-shifted softmax on logits `[1000,1001,999]`. The naive exponential can overflow. An overflow warning is evidence of a numerical problem, not a harmless message to suppress. Then add a constant of 10,000 to every logit and verify the stable probabilities do not change substantially. This checks a mathematical invariance independently of a specific expected probability vector.



In [ ]:
import numpy as np
def stable_softmax(z):
    z = np.asarray(z, dtype=np.float64)
    shifted = z-z.max()
    return np.exp(shifted)/np.exp(shifted).sum()
p = stable_softmax([1000., 1001., 999.])
q = stable_softmax([11000., 11001., 10999.])
np.testing.assert_allclose(p, q)
np.testing.assert_allclose(p.sum(), 1.)



## 6. From observations to a report

Your report should contain a question, the relevant equation and assumptions, the implementation decision, a table or plot of actual results, an interpretation and a limitation. A suitable limitation is that finite-difference agreement was tested only at particular smooth points in float64. An unsuitable conclusion is “autograd is always correct” or “ridge always improves prediction,” because the experiment did not test those universal claims.

Keep demonstration outputs separate from the assessed variations. Copying the bundle's default results proves that the script ran, not that you can design a check. The module assignment requires at least one new shape example, one new gradient point, a conditioning perturbation and a paired comparison that you explain yourself.

## Exercises

1. Replace the correct gradient's averaging factor `1/n` with `1/d`. Predict when this bug is hidden, then choose a test where it is visible.
2. Construct a finite-difference test at a ReLU kink and explain why failure is inconclusive there.
3. Change bootstrap seed while keeping the original paired observations fixed. Separate Monte Carlo variability from data uncertainty in your explanation.
4. Add a test for a zero-vector cosine input. State whether your API rejects it or returns an explicitly documented sentinel.

## Self-check and readiness gate

For 1, the mistake is hidden when n equals d; choose unequal dimensions and nonzero gradients. For 2, the classical derivative is undefined at the kink and central differences can cross both branches. For 3, only the resampling computation changes, not the original evidence. For 4, silently returning a meaningful-looking cosine for an undefined case is poor documentation; a clear error is reasonable.

You are ready to submit when a peer can run your command in a fresh environment, see the same qualitative conclusions, inspect the raw metrics and locate the tests that would catch your chosen failure cases. Continue to the module assignment in the portal for deliverables and the 100-point rubric.

## Primary reading

- [NumPy: numerical `allclose` comparison](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html): read the tolerance equation and the warning about small reference values.
- [Goodfellow, Bengio and Courville: Numerical Computation](https://www.deeplearningbook.org/contents/numerical.html): connect overflow, underflow and conditioning to the concrete failures you observed.
